# 01 - Inspeksi dataset tomato, pumpkin, lemon

Notebook ini hanya membaca dataset. Tidak ada training dan tidak ada file dataset yang diubah.

In [ ]:
from collections import Counter
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from PIL import Image

PACKAGE = Path.cwd()
PACKAGE = next((p for p in (PACKAGE, *PACKAGE.parents) if (p / 'dataset' / 'data.yaml').is_file()), None) or next((p.parent.parent for p in Path.cwd().glob('**/dataset/data.yaml')), None)
if PACKAGE is None:
    raise FileNotFoundError('Folder paket tomato-pumpkin-lemon-training belum dibuka.')
sys.path.insert(0, str(PACKAGE / 'notebooks'))
sys.path.insert(0, str(PACKAGE / 'training'))
from notebook_helpers import find_package_root
from dataset_validation import validate_dataset

PACKAGE = find_package_root(PACKAGE)
DATASET = PACKAGE / 'dataset'
DATA_YAML = PACKAGE / 'dataset/data.yaml'
config = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
CLASS_NAMES = list(config['names'])
print('Package:', PACKAGE)
print('Kelas:', CLASS_NAMES)


In [ ]:
splits = ('train', 'val', 'test', 'external_holdout')
for split in splits:
    images = list((DATASET / 'images' / split).glob('*'))
    labels = list((DATASET / 'labels' / split).glob('*.txt'))
    print(f'{split:16} images={len(images):4} labels={len(labels):4}')

validation = validate_dataset(DATASET)
print('Status validasi:', validation['status'])
print('Invalid label:', validation['invalid_label_count'])
print('Synthetic image:', validation['synthetic_images'])
print('Jumlah objek:', validation['class_counts'])


In [ ]:
def read_boxes(label_path):
    boxes = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        values = line.split()
        if len(values) == 5:
            boxes.append([int(values[0]), *map(float, values[1:])])
    return boxes

def show_examples(split='train', per_class=2, seed=20261124):
    random.seed(seed)
    by_class = {index: [] for index in range(len(CLASS_NAMES))}
    for label in sorted((DATASET / 'labels' / split).glob('*.txt')):
        boxes = read_boxes(label)
        for box in boxes:
            by_class[box[0]].append((label, box))
    for class_id, entries in by_class.items():
        selected = random.sample(entries, min(per_class, len(entries)))
        fig, axes = plt.subplots(1, max(1, len(selected)), figsize=(12, 4))
        axes = [axes] if len(selected) == 1 else axes
        for axis, (label, box) in zip(axes, selected):
            image_path = DATASET / 'images' / split / f'{label.stem}.jpg'
            if not image_path.is_file():
                image_path = next((p for p in (DATASET / 'images' / split).glob(f'{label.stem}.*')), None)
            image = Image.open(image_path).convert('RGB')
            axis.imshow(image)
            axis.set_title(CLASS_NAMES[class_id])
            axis.axis('off')
        fig.suptitle(f'{split}: {CLASS_NAMES[class_id]}')
        plt.show()

show_examples()
